In [ ]:
# Uninstall the currently installed version of the httpx library
# httpx is a Python library used to send HTTP requests and receive responses from websites, APIs, and online services.
!pip uninstall -y httpx

# Install a specific version of httpx (0.25.2)
!pip install httpx==0.25.2

In [ ]:
# Uninstall the currently installed LangChain packages
!pip uninstall -y langchain langchain-core langchain-community

# Install a specific version of LangChain (0.0.353)
!pip install langchain==0.0.353

# Install the llama-cpp-python library for running GGUF LLMs locally
!pip install llama-cpp-python

In [ ]:
# Download the Phi-3 Mini 4K Instruct GGUF model from Hugging Face
!wget https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf

In [ ]:
# Import the LlamaCpp class from LangChain
from langchain import LlamaCpp

# Load the local GGUF language model
llm = LlamaCpp(

    # Path to the GGUF model file
    model_path="Phi-3-mini-4k-instruct-fp16.gguf",

    # Number of model layers to run on the GPU (-1 = all layers)
    n_gpu_layers=-1,

    # Maximum number of tokens the model can generate
    max_tokens=500,

    # Maximum context window (input + output tokens)
    n_ctx=2048,

    # Set a random seed for reproducible outputs
    seed=42,

    # Disable detailed execution logs
    verbose=False
)

In [ ]:
# Send a prompt to the loaded LlamaCpp model
response = llm.invoke("Hi! My name is Maarten. What is 1 + 1?")

# Print the model's response
print(response)

In [ ]:
# Import the ChatOpenAI class from LangChain
from langchain.chat_models import ChatOpenAI

# Create a chat-based Large Language Model using the OpenAI API
# Replace "" with your OpenAI API key before running
chat_model = ChatOpenAI(
    openai_api_key="YOUR_API_KEY"
)

In [ ]:
# Import the PromptTemplate class from LangChain
from langchain import PromptTemplate

# Define the prompt template with a placeholder variable
template = """<s><|user|>
{input_prompt}
<|end|>
<|assistant|>"""

# Create a PromptTemplate object
prompt = PromptTemplate(

    # Specify the prompt format
    template=template,

    # Define the variable that will be replaced with user input
    input_variables=["input_prompt"]
)

In [ ]:
# Create a LangChain pipeline by connecting the prompt template to the LLM
basic_chain = prompt | llm

In [ ]:
# Run the LangChain pipeline using the given input
basic_chain.invoke(

    # Pass the values for the prompt template variables
    {
        # Replace {input_prompt} in the PromptTemplate with this text
        "input_prompt": "Hi! My name is Maarten. What is 1 + 1?",
    }
)

In [ ]:
# Create a prompt template for generating a business name
template = "Create a funny name for a business that sells {product}."

# Create a PromptTemplate object
name_prompt = PromptTemplate(

    # Define the prompt format
    template=template,

    # Specify the variable that will be replaced in the template
    input_variables=["product"]
)

In [ ]:
# Import the LLMChain class from LangChain
from langchain import LLMChain

# Create a prompt template for generating a story title
template = """<s><|user|>
Create a title for a story about
{summary}
. Only return the title.
<|end|>
<|assistant|>"""

# Create a PromptTemplate with the variable 'summary'
title_prompt = PromptTemplate(
    template=template,
    input_variables=["summary"]
)

# Create an LLMChain that connect prompt template and the LLM
title = LLMChain(

    # Language model used to generate the response
    llm=llm,

    # Prompt template used by the model
    prompt=title_prompt,

    # Name of the output produced by this chain
    output_key="title"
)

In [ ]:
# Run the title generation chain
title.invoke({

    # Pass the story summary to the PromptTemplate
    "summary": "a girl that lost her mother"

})

In [ ]:
# Create a prompt template for generating the main character description
template = """<s><|user|>
Describe the main character of a story about
{summary}
with the title
{title}.
Use only two sentences.
<|end|>
<|assistant|>"""

# Create a PromptTemplate with two input variables
character_prompt = PromptTemplate(
    template=template,
    input_variables=["summary", "title"]
)

# Create an LLMChain for generating the character description
character = LLMChain(

    # Language model used to generate the response
    llm=llm,

    # Prompt template used by this chain
    prompt=character_prompt,

    # Store the generated output with the key "character"
    output_key="character"
)

In [ ]:
# Create a prompt template for generating the complete story
template = """<s><|user|>
Create a story about
{summary}
with the title
{title}.
The main character is:
{character}.
Only return the story and it cannot be longer than one paragraph.
<|end|>
<|assistant|>"""

# Create a PromptTemplate with three input variables
story_prompt = PromptTemplate(
    template=template,
    input_variables=["summary", "title", "character"]
)

# Create an LLMChain for generating the story
story = LLMChain(

    # Language model used to generate the story
    llm=llm,

    # Prompt template used by this chain
    prompt=story_prompt,

    # Store the generated output with the key "story"
    output_key="story"
)

In [ ]:
# Combine all three components to create the full chain
llm_chain = title | character | story

In [ ]:
# Run the complete LangChain pipeline
llm_chain.invoke(

    # Input provided to the first chain (story summary)
    "a girl that lost her mother"

)

In [ ]:
# Run the basic LangChain pipeline
basic_chain.invoke(

    # Pass the value for the 'input_prompt' variable in the PromptTemplate
    {
        "input_prompt": "Hi! My name is Maarten. What is 1 + 1?"
    }

)

In [ ]:
# Run the LangChain pipeline with a new question
basic_chain.invoke(

    # Pass the question as the input for the PromptTemplate
    {
        "input_prompt": "What is my name?"
    }

)

In [ ]:
# Create a prompt template that includes previous conversation history
template = """<s><|user|>
Current conversation:
{chat_history}

{input_prompt}
<|end|>
<|assistant|>"""

# Create a PromptTemplate with two input variables
prompt = PromptTemplate(

    # Define the prompt format
    template=template,

    # Specify the variables that will be replaced in the template
    input_variables=["input_prompt", "chat_history"]
)

In [ ]:
# Import the ConversationBufferMemory class
from langchain.memory import ConversationBufferMemory

# Create a memory object to store the conversation history
memory = ConversationBufferMemory(
    memory_key="chat_history"
)

# Create an LLMChain using the prompt, language model, and memory
llm_chain = LLMChain(

    # Prompt template
    prompt=prompt,

    # Language model
    llm=llm,

    # Conversation memory
    memory=memory
)

In [ ]:
# Generate a conversation and ask a basic question
llm_chain.invoke({"input_prompt": "Hi! My name is Savin. What is 1 + 1?"})

In [ ]:
# Checj does the LLM remember the name we gave it?
llm_chain.invoke({"input_prompt": "What is my name?"})

In [ ]:
# Import ConversationBufferWindowMemory
from langchain.memory import ConversationBufferWindowMemory

# Create memory that stores only the last 2 conversations
memory = ConversationBufferWindowMemory(
    k=2,
    memory_key="chat_history"
)

# Create an LLMChain using the prompt, model, and window memory
llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory
)

# First conversation
llm_chain.predict(
    input_prompt="Hi! My name is Savin and I am 19 years old. What is 1 + 1?"
)

# Second conversation
llm_chain.predict(
    input_prompt="What is 3 + 3?"
)

In [ ]:
# Check whether it knows the name we gave it
llm_chain.invoke({"input_prompt":"What is my name?"})

In [ ]:
# Check whether it knows the age we gave it
llm_chain.invoke({"input_prompt":"What is my age?"})

In [ ]:
# Create a prompt template for summarizing the conversation
summary_prompt_template = """<s><|user|>
Summarize the conversations and update with the new lines.

Current summary:
{summary}

New lines of conversation:
{new_lines}

New summary:
<|end|>
<|assistant|>"""

# Create a PromptTemplate with two input variables
summary_prompt = PromptTemplate(

    # Variables to replace in the template
    input_variables=["new_lines", "summary"],

    # Prompt template
    template=summary_prompt_template
)

In [ ]:
# Import ConversationSummaryMemory
from langchain.memory import ConversationSummaryMemory

# Create a memory object that stores a summarized conversation
memory = ConversationSummaryMemory(

    # LLM used to generate/update the conversation summary
    llm=llm,

    # Variable name used in the PromptTemplate
    memory_key="chat_history",

    # Prompt used to summarize the conversation
    prompt=summary_prompt
)

# Create an LLMChain using the prompt, language model, and summary memory
llm_chain = LLMChain(

    # Prompt template for the chatbot
    prompt=prompt,

    # Language model
    llm=llm,

    # Conversation summary memory
    memory=memory
)

In [ ]:
# Generate a conversation and ask for the name
llm_chain.invoke({"input_prompt": "Hi! My name is Savin . What is 1 + 1?"})
llm_chain.invoke({"input_prompt": "What is my name?"})

In [ ]:
# Check whether it has summarized everything thus far
llm_chain.invoke({"input_prompt": "What was the first question I asked?"})

In [ ]:
# Check what the summary is thus far
memory.load_memory_variables({})

Below code requires a paid OpenAI API key to run

In [ ]:
# Uninstall old versions of LangChain packages
!pip uninstall -y langchain langchain-core langchain-community

# Install the latest versions of LangChain, LangChain OpenAI integration, and OpenAI SDK
!pip install -U langchain langchain-openai openai

In [ ]:
# Import the os module to work with environment variables
import os

# Import the ChatOpenAI class from LangChain
from langchain_openai import ChatOpenAI

# Store your OpenAI API key as an environment variable
os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY"

# Create a ChatOpenAI language model
openai_llm = ChatOpenAI(

    # Specify the OpenAI model to use
    model_name="gpt-3.5-turbo",

    # Set temperature to 0 for deterministic and consistent responses
    temperature=0
)

In [ ]:
# Create a ReAct prompt template for the AI agent
react_template = """Answer the following questions as best you can.

# List all available tools
You have access to the following tools:
{tools}

# Specify the reasoning format the agent must follow
Use the following format:

Question: the input question you must answer

Thought: you should always think about what to do

Action: the action to take, should be one of [{tool_names}]

Action Input: the input to the action

Observation: the result of the action

... (this Thought/Action/Action Input/Observation can repeat N times)

Thought: I now know the final answer

Final Answer: the final answer to the original input question

Begin!

Question:
{input}

Thought:
{agent_scratchpad}
"""

# Create the PromptTemplate with the required variables
prompt = PromptTemplate(

    # Prompt template
    template=react_template,

    # Variables that will be replaced before sending the prompt to the LLM
    input_variables=[
        "tools",
        "tool_names",
        "input",
        "agent_scratchpad"
    ]
)

In [ ]:
# Import functions to load built-in tools and create custom tools
from langchain.agents import load_tools, Tool

# Import the DuckDuckGo web search tool
# This tool allows the agent to search the web.
from langchain.tools import DuckDuckGoSearchResults

# Create a DuckDuckGo search object
search = DuckDuckGoSearchResults()

# Convert the search object into a LangChain Tool
search_tool = Tool(

    # Name of the tool
    name="duckduck",

    # Description used by the agent to decide when to use this tool
    description="A web search engine. Use this as a search engine for general queries.",

    # Function that executes the web search
    func=search.run,
)

# Load the built-in mathematical calculation tool
tools = load_tools(
    ["llm-math"],
    llm=openai_llm
)

# Add the custom DuckDuckGo search tool to the list of tools
tools.append(search_tool)

In [ ]:
# Import the AgentExecutor and ReAct agent creation function
from langchain.agents import AgentExecutor, create_react_agent

# Create a ReAct agent using the LLM, available tools, and prompt template
# Creates a ReAct (Reason + Action ) Agent.
agent = create_react_agent(
    openai_llm,
    tools,
    prompt
)

# Create an AgentExecutor to run the ReAct agent
agent_executor = AgentExecutor(

    # The ReAct agent
    agent=agent,

    # List of tools available to the agent
    tools=tools,

    # Display the agent's reasoning process
    verbose=True,

    # Continue execution even if output parsing errors occur
    handle_parsing_errors=True
)

In [ ]:
# What is the price of a MacBook Pro?
agent_executor.invoke(
{
"input": "What is the current price of a MacBook Pro in USD? How much would it cost in EUR if the exchange rate is 0.85 EUR for 1 USD."
}
)